# Local Agentic AI with Gemma 3 — Lab 2: Pydantic & Tool Calling

In this lab, we upgrade our local **Gemma 3 (4B)** model from a conversational LLM into an **autonomous agent** capable of invoking external tools. Autonomous agents rely on structured data contracts to interface reliably with real-world APIs, databases, and calculation engines without hallucination.

| Interaction Pattern | Structured Schema | Deterministic Types | Hallucination Risk | Best Use Case |
|---|---|---|---|---|
| **Raw Prompting** | ❌ None | ❌ Strings only | ⚠️ High | General chat & drafting |
| **Regex Extraction** | ⚠️ Fragile | ⚠️ Manual parsing | ⚠️ Moderate | Legacy text extraction |
| **Pydantic Schemas** | ✅ JSON Schema | ✅ Strict Type Checking | 🛡️ Minimal | **Autonomous Agent Tool Calling** |

> **Core Philosophy:** Real-world tools require guaranteed data types. By pairing **Pydantic** validation models with Ollama's constrained decoding (`format=schema`), we ensure that Gemma 3 emits 100% valid, type-safe parameters every single invocation.

## 1. Prerequisites & Agent Execution Architecture

Our autonomous agent operates via a closed-loop execution pattern:
1. **Intent Analysis & Tool Routing**: The model evaluates the user prompt against a Pydantic schema and selects either a tool (`get_weather`, `calculator`) or direct conversation (`none`).
2. **Parameter Validation**: Pydantic validates incoming parameters at runtime.
3. **Tool Dispatcher**: The Python environment executes the selected function and captures deterministic outputs.
4. **Final Synthesis**: Gemma 3 formats the tool findings into a user-friendly Markdown response.

In [1]:
# ── Library Imports & Client Setup ──────────────────────────────────────────
from pydantic import BaseModel, Field, ConfigDict
from typing import Optional, Literal, Dict, Any
import json
import ollama
from IPython.display import Markdown, display

# Initialize local Ollama client connected to localhost:11434
client = ollama.Client(host="http://localhost:11434")
MODEL_NAME = "gemma3:4b"

print(f"✅ Ollama client connected to {client._client.base_url}")
print(f"  Target Model : {MODEL_NAME}")

✅ Ollama client connected to http://localhost:11434/
  Target Model : gemma3:4b


## 2. Pydantic Fundamentals: Runtime Validation & Type Coercion

Pydantic models enforce data structure and type integrity at runtime. In the code cell below, notice how Pydantic automatically coerces string numbers into integers while strictly validating minimum lengths and age boundaries.

In [2]:
# Define a sample Pydantic validation model
class UserProfile(BaseModel):
    user_id: int = Field(..., description="Unique user identifier")
    name: str = Field(..., min_length=2, description="Full user name")
    email: str = Field(..., description="Contact email address")
    age: Optional[int] = Field(None, ge=18, le=120, description="Age between 18 and 120")

# Demonstration of automatic type coercion ('101' -> 101, '28' -> 28)
user = UserProfile(user_id="101", name="Alex Morgan", email="alex@example.com", age="28")
print("Validated User Object (Dictionary representation):")
print(json.dumps(user.model_dump(), indent=2))

Validated User Object (Dictionary representation):
{
  "user_id": 101,
  "name": "Alex Morgan",
  "email": "alex@example.com",
  "age": 28
}


## 3. Defining Strongly Typed Tool Schemas

Instead of duplicating tool definitions, we define **two distinct, complementary tools**:
1. `WeatherToolParams`: Environmental telemetry tool requiring a city name and temperature unit (`celsius` or `fahrenheit`).
2. `CalculatorToolParams`: Arithmetic computation tool requiring a mathematical expression string.

In [3]:
# Tool 1 Schema: Weather Telemetry
class WeatherToolParams(BaseModel):
    """Retrieve current weather conditions, temperature, and attire recommendations."""
    location: str = Field(..., description="The target city, e.g. 'Tokyo' or 'New York'")
    unit: Literal["celsius", "fahrenheit"] = Field("celsius", description="Temperature scale")

# Tool 2 Schema: Mathematical Calculation
class CalculatorToolParams(BaseModel):
    """Perform precise arithmetic calculations and evaluations."""
    expression: str = Field(..., description="Math expression to calculate, e.g. '25 * 48'")

print("✓ Tool schemas defined successfully.")
print("Generated Weather JSON Schema:")
print(json.dumps(WeatherToolParams.model_json_schema(), indent=2))

✓ Tool schemas defined successfully.
Generated Weather JSON Schema:
{
  "description": "Retrieve current weather conditions, temperature, and attire recommendations.",
  "properties": {
    "location": {
      "description": "The target city, e.g. 'Tokyo' or 'New York'",
      "title": "Location",
      "type": "string"
    },
    "unit": {
      "default": "celsius",
      "description": "Temperature scale",
      "enum": [
        "celsius",
        "fahrenheit"
      ],
      "title": "Unit",
      "type": "string"
    }
  },
  "required": [
    "location"
  ],
  "title": "WeatherToolParams",
  "type": "object"
}


## 4. Unified Agent Routing Contract

To give our agent full autonomous control over when to call a tool versus when to answer directly, we combine these parameters into a unified decision contract: `AgentToolDecision`. Gemma 3 will output strictly conforming JSON matching this schema.

In [4]:
class AgentToolDecision(BaseModel):
    """Unified decision schema for agent tool selection and parameterization."""
    tool: Literal["get_weather", "calculator", "none"] = Field(
        ..., description="Selected tool name, or 'none' if no tool is required"
    )
    reasoning: str = Field(..., description="Step-by-step reasoning for this decision")
    weather_params: Optional[WeatherToolParams] = Field(None, description="Parameters if get_weather is selected")
    calculator_params: Optional[CalculatorToolParams] = Field(None, description="Parameters if calculator is selected")

agent_schema = AgentToolDecision.model_json_schema()
print("✓ Unified Agent Decision Schema ready for Ollama constrained decoding.")

✓ Unified Agent Decision Schema ready for Ollama constrained decoding.


## 5. Tool Implementation & Dispatcher Registry

Now we write the actual Python functions that execute when called. We avoid external API key dependencies by providing a reliable local telemetry service and a safe math evaluator.

In [5]:
# 1. Weather Telemetry Service
def get_weather_data(location: str, unit: str = "celsius") -> Dict[str, Any]:
    city = location.split(",")[0].strip().title()
    database = {
        "Tokyo": {"temp_c": 19, "condition": "Partly Cloudy", "humidity": "62%", "clothing": "Light jacket"},
        "New York": {"temp_c": 22, "condition": "Sunny", "humidity": "45%", "clothing": "T-shirt and sunglasses"},
        "London": {"temp_c": 14, "condition": "Light Rain", "humidity": "80%", "clothing": "Raincoat and umbrella"},
        "Toronto": {"temp_c": 16, "condition": "Breezy", "humidity": "55%", "clothing": "Sweater or windbreaker"}
    }
    record = database.get(city, {"temp_c": 20, "condition": "Clear", "humidity": "50%", "clothing": "Casual wear"})
    temperature = record["temp_c"] if unit == "celsius" else round(record["temp_c"] * 9/5 + 32, 1)
    
    return {
        "location": city,
        "temperature": f"{temperature}°{'C' if unit == 'celsius' else 'F'}",
        "condition": record["condition"],
        "humidity": record["humidity"],
        "recommended_attire": record["clothing"]
    }

# 2. Safe Calculator Service
def calculate_expression(expression: str) -> Dict[str, Any]:
    try:
        allowed_chars = set("0123456789+-*/(). % ")
        if not all(c in allowed_chars for c in expression):
            raise ValueError("Contains disallowed characters")
        result = eval(expression, {"__builtins__": None}, {})
        return {"expression": expression, "result": result, "status": "success"}
    except Exception as e:
        return {"expression": expression, "error": str(e), "status": "error"}

# Central Tool Dispatcher Registry
TOOL_REGISTRY = {
    "get_weather": lambda p: get_weather_data(p.location, p.unit),
    "calculator": lambda p: calculate_expression(p.expression)
}

print(f"✓ Tool Dispatcher registered {len(TOOL_REGISTRY)} active tools: {list(TOOL_REGISTRY.keys())}")

✓ Tool Dispatcher registered 2 active tools: ['get_weather', 'calculator']


## 6. Autonomous Agent Execution Loop

The `run_agent` function coordinates the two-phase loop:
1. **Phase 1 (Routing)**: Gemma 3 analyzes the query and returns an `AgentToolDecision`.
2. **Phase 2 (Dispatch & Synthesis)**: Python executes the matched tool and returns the observation back to Gemma 3 to generate the final response.

In [6]:
def run_agent(user_query: str) -> str:
    print(f"\n💬 User Query: \"{user_query}\"")
    
    # Phase 1: Structured Decision with Pydantic JSON Schema
    response = client.chat(
        model=MODEL_NAME,
        messages=[
            {
                "role": "system",
                "content": "You are an autonomous AI agent with access to external tools. Analyze the user request. When selecting a tool, you must populate the corresponding params object (weather_params for get_weather, calculator_params for calculator). If no tool is needed, set tool to 'none'."
            },
            {"role": "user", "content": user_query}
        ],
        format=agent_schema
    )
    
    # Validate structured JSON decision with Pydantic
    decision = AgentToolDecision.model_validate_json(response["message"]["content"])
    print(f"🤖 Agent Reasoning: {decision.reasoning}")
    print(f"🔧 Tool Selected  : {decision.tool}")
    
    if decision.tool == "none":
        return decision.reasoning
        
    # Phase 2: Execute matched tool
    if decision.tool == "get_weather":
        tool_output = TOOL_REGISTRY["get_weather"](decision.weather_params)
    elif decision.tool == "calculator":
        tool_output = TOOL_REGISTRY["calculator"](decision.calculator_params)
    
    print(f"⚙️ Tool Output    : {tool_output}")
    
    # Phase 3: Final Response Synthesis
    synth_response = client.chat(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "You are an articulate technical assistant. Formulate a clean, formatted Markdown answer to the user query based on the tool result."},
            {"role": "user", "content": user_query},
            {"role": "assistant", "content": f"Tool execution result: {json.dumps(tool_output)}"},
            {"role": "user", "content": "Please present the final answer to my question based on this data."}
        ]
    )
    
    return synth_response["message"]["content"]

print("✓ Autonomous Agent loop compiled and ready for inference.")


✓ Autonomous Agent loop compiled and ready for inference.


## 7. Multi-Scenario Evaluation

We now test our agent across three fundamental scenarios:
1. **Weather Lookup**: Requests real-time environmental telemetry for Tokyo.
2. **Arithmetic Calculation**: Requires exact multiplication (`25 * 48`).
3. **Conversational Query**: General knowledge question requiring zero tool invocations.

In [7]:
# Scenario 1: Weather Telemetry Lookup
ans1 = run_agent("What is the weather in Tokyo in celsius?")
display(Markdown(f"### Scenario 1 Answer:\n{ans1}"))

# Scenario 2: Precise Arithmetic Calculation
ans2 = run_agent("Calculate 25 * 48.")
display(Markdown(f"### Scenario 2 Answer:\n{ans2}"))

# Scenario 3: Pure Conversational Intent
ans3 = run_agent("What is the capital of France?")
display(Markdown(f"### Scenario 3 Answer:\n{ans3}"))

💬 User Query: "What is the weather in Tokyo in celsius?"
🤖 Agent Reasoning: I need to find the weather in Tokyo and express it in Celsius.
🔧 Tool Selected  : get_weather
⚙️ Tool Output    : {'location': 'Tokyo', 'temperature': '19°C', 'condition': 'Partly Cloudy', 'humidity': '62%', 'recommended_attire': 'Light jacket'}

💬 User Query: "Calculate 25 * 48."
🤖 Agent Reasoning: I need to perform a multiplication operation. The numbers are 25 and 48.
🔧 Tool Selected  : calculator
⚙️ Tool Output    : {'expression': '25 * 48', 'result': 1200, 'status': 'success'}

💬 User Query: "What is the capital of France?"
🤖 Agent Reasoning: This is a factual question that can be answered directly with common knowledge.
🔧 Tool Selected  : none


### Scenario 1 Answer:
Okay, here’s the weather information for Tokyo:

*   **Temperature:** 19°C (Partly Cloudy)
*   **Humidity:** 62%
*   **Recommended Attire:** Light jacket 

Do you want any further weather details, such as the forecast for the next few days?


### Scenario 2 Answer:
Okay, the calculation of 25 * 48 results in **1200**.


### Scenario 3 Answer:
This is a factual question that can be answered directly with common knowledge.
